# BreastDM UNeXt — Improved Generalization Experiment

This self-contained Colab notebook keeps the leak-free patient split and improves the earlier baseline with paper-aligned **224×224 grayscale input**, conservative augmentation, AdamW, warm-up plus cosine decay, mixed precision, early stopping, recovery checkpoints, validation-only threshold selection, and both per-image and global foreground metrics. Run every cell in order.

In [ ]:
%pip install -q albumentations


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import csv, json, random, shutil, time
from pathlib import Path
import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 100
PATIENCE = 20
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'

SOURCE_ZIP = Path('/content/drive/MyDrive/BreastDM_Project/data/segmentation_DS_7_14_2026.zip')
DATASET_ROOT = Path('/content/segmentation_DS_7_14_2026')
RUN_DIR = Path('/content/drive/MyDrive/BreastDM_Project/UNeXt_runs/BreastDM_UNeXt_Improved_224_Gray')
RUN_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = RUN_DIR / 'best_model.pt'
RECOVERY_PATH = RUN_DIR / 'recovery_checkpoint.pt'
HISTORY_PATH = RUN_DIR / 'training_history.csv'
RESULTS_PATH = RUN_DIR / 'test_results.json'

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
assert DEVICE.type == 'cuda', 'Enable a Colab GPU before training.'
if not DATASET_ROOT.is_dir():
    assert SOURCE_ZIP.is_file(), f'Missing dataset ZIP: {SOURCE_ZIP}'
    shutil.unpack_archive(str(SOURCE_ZIP), '/content')
print('Device:', DEVICE, '| AMP:', USE_AMP, '| Run:', RUN_DIR)


In [ ]:
class BreastDMSegmentationDataset(Dataset):
    def __init__(self, root, split, transform=None):
        if split not in {'train', 'val', 'test'}: raise ValueError(split)
        self.split, self.transform = split, transform
        image_dir, mask_dir = Path(root)/split/'images', Path(root)/split/'masks'
        images = {p.stem:p for p in image_dir.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png'}}
        masks = {p.stem:p for p in mask_dir.iterdir() if p.suffix.lower() in {'.png','.jpg','.jpeg'}}
        missing_masks, missing_images = set(images)-set(masks), set(masks)-set(images)
        if missing_masks or missing_images:
            raise ValueError(f'Pairing error: {len(missing_masks)} missing masks, {len(missing_images)} missing images')
        self.samples = [(images[s], masks[s], s) for s in sorted(images)]
        if not self.samples: raise ValueError(f'No samples in {split}')

    def __len__(self): return len(self.samples)

    @property
    def patient_ids(self): return {s.split('__', 1)[0] for _,_,s in self.samples}

    def __getitem__(self, index):
        image_path, mask_path, sample_id = self.samples[index]
        image = np.asarray(Image.open(image_path).convert('L'), dtype=np.uint8)
        mask = np.asarray(Image.open(mask_path).convert('L'), dtype=np.uint8)
        if image.shape != mask.shape: raise ValueError(f'Shape mismatch: {sample_id}')
        if self.transform is not None:
            transformed = self.transform(image=image, mask=mask)
            image, mask = transformed['image'], transformed['mask']
        image = torch.from_numpy(np.asarray(image).copy()).float().unsqueeze(0) / 255.0
        mask = (torch.from_numpy(np.asarray(mask).copy()).float() > 0).float().unsqueeze(0)
        patient_id = sample_id.split('__', 1)[0]
        return image, mask, {'img_id':sample_id, 'patient_id':patient_id, 'split':self.split}

train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=0.2),
])
eval_transform = A.Compose([A.Resize(IMAGE_SIZE, IMAGE_SIZE)])

train_dataset = BreastDMSegmentationDataset(DATASET_ROOT, 'train', train_transform)
val_dataset = BreastDMSegmentationDataset(DATASET_ROOT, 'val', eval_transform)
test_dataset = BreastDMSegmentationDataset(DATASET_ROOT, 'test', eval_transform)
assert not (train_dataset.patient_ids & val_dataset.patient_ids)
assert not (train_dataset.patient_ids & test_dataset.patient_ids)
assert not (val_dataset.patient_ids & test_dataset.patient_ids)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed); np.random.seed(worker_seed)

generator = torch.Generator().manual_seed(SEED)
common = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS>0, worker_init_fn=seed_worker)
train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, **common)
val_loader = DataLoader(val_dataset, shuffle=False, **common)
test_loader = DataLoader(test_dataset, shuffle=False, **common)

images, masks, meta = next(iter(train_loader))
assert images.shape[1:] == (1, IMAGE_SIZE, IMAGE_SIZE) and masks.shape == images.shape
print('Samples:', len(train_dataset), len(val_dataset), len(test_dataset))
print('Batch:', images.shape, masks.shape, '| patients:', len(train_dataset.patient_ids), len(val_dataset.patient_ids), len(test_dataset.patient_ids))


In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.trunc_normal_(m.weight, std=.02)
        if m.bias is not None: nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LayerNorm): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Conv2d):
        fan = m.kernel_size[0]*m.kernel_size[1]*m.out_channels//m.groups
        nn.init.normal_(m.weight, 0, (2.0/fan)**0.5)
        if m.bias is not None: nn.init.zeros_(m.bias)

class DWConv(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.conv = nn.Conv2d(dim, dim, 3, 1, 1, groups=dim)
    def forward(self, x, h, w):
        b,n,c=x.shape; x=x.transpose(1,2).reshape(b,c,h,w); x=self.conv(x)
        return x.flatten(2).transpose(1,2)

class ShiftMLP(nn.Module):
    def __init__(self, dim, shift_size=5):
        super().__init__(); self.shift_size=shift_size; self.pad=shift_size//2
        self.fc1=nn.Linear(dim,dim); self.dw=DWConv(dim); self.act=nn.GELU(); self.fc2=nn.Linear(dim,dim)
        self.apply(init_weights)
    def shift(self, x, h, w, spatial_dim):
        x=F.pad(x,(self.pad,)*4); groups=torch.chunk(x,self.shift_size,dim=1)
        x=torch.cat([torch.roll(g,s,spatial_dim) for g,s in zip(groups,range(-self.pad,self.pad+1))],dim=1)
        return x[:,:,self.pad:self.pad+h,self.pad:self.pad+w]
    def forward(self,x,h,w):
        b,n,c=x.shape; x=x.transpose(1,2).reshape(b,c,h,w); x=self.shift(x,h,w,2)
        x=x.reshape(b,c,h*w).transpose(1,2); x=self.act(self.dw(self.fc1(x),h,w))
        x=x.transpose(1,2).reshape(b,c,h,w); x=self.shift(x,h,w,3)
        return self.fc2(x.reshape(b,c,h*w).transpose(1,2))

class ShiftedBlock(nn.Module):
    def __init__(self,dim):
        super().__init__(); self.norm=nn.LayerNorm(dim); self.mlp=ShiftMLP(dim); self.apply(init_weights)
    def forward(self,x,h,w): return x+self.mlp(self.norm(x),h,w)

class PatchEmbed(nn.Module):
    def __init__(self,in_channels,dim):
        super().__init__(); self.proj=nn.Conv2d(in_channels,dim,3,2,1); self.norm=nn.LayerNorm(dim); self.apply(init_weights)
    def forward(self,x):
        x=self.proj(x); _,_,h,w=x.shape; return self.norm(x.flatten(2).transpose(1,2)),h,w

class UNext(nn.Module):
    def __init__(self,input_channels=1,num_classes=1):
        super().__init__()
        self.e1=nn.Conv2d(input_channels,8,3,1,1); self.e2=nn.Conv2d(8,16,3,1,1); self.e3=nn.Conv2d(16,32,3,1,1)
        self.eb1=nn.BatchNorm2d(8); self.eb2=nn.BatchNorm2d(16); self.eb3=nn.BatchNorm2d(32)
        self.p3=PatchEmbed(32,64); self.p4=PatchEmbed(64,128)
        self.b1=ShiftedBlock(64); self.b2=ShiftedBlock(128); self.n3=nn.LayerNorm(64); self.n4=nn.LayerNorm(128)
        self.d1=nn.Conv2d(128,64,3,1,1); self.d2=nn.Conv2d(64,32,3,1,1); self.d3=nn.Conv2d(32,16,3,1,1); self.d4=nn.Conv2d(16,8,3,1,1); self.d5=nn.Conv2d(8,8,3,1,1)
        self.db1=nn.BatchNorm2d(64); self.db2=nn.BatchNorm2d(32); self.db3=nn.BatchNorm2d(16); self.db4=nn.BatchNorm2d(8)
        self.dbk1=ShiftedBlock(64); self.dbk2=ShiftedBlock(32); self.dn3=nn.LayerNorm(64); self.dn4=nn.LayerNorm(32)
        self.final=nn.Conv2d(8,num_classes,1)
    @staticmethod
    def fmap(x,b,h,w): return x.reshape(b,h,w,-1).permute(0,3,1,2).contiguous()
    @staticmethod
    def up(x,target): return F.interpolate(x,size=target.shape[-2:],mode='bilinear',align_corners=False)
    def forward(self,x):
        b=x.shape[0]; original=x.shape[-2:]
        x=F.relu(F.max_pool2d(self.eb1(self.e1(x)),2)); t1=x
        x=F.relu(F.max_pool2d(self.eb2(self.e2(x)),2)); t2=x
        x=F.relu(F.max_pool2d(self.eb3(self.e3(x)),2)); t3=x
        x,h,w=self.p3(x); x=self.fmap(self.n3(self.b1(x,h,w)),b,h,w); t4=x
        x,h,w=self.p4(x); x=self.fmap(self.n4(self.b2(x,h,w)),b,h,w)
        x=F.relu(self.up(self.db1(self.d1(x)),t4))+t4; h,w=x.shape[-2:]; x=x.flatten(2).transpose(1,2)
        x=self.fmap(self.dn3(self.dbk1(x,h,w)),b,h,w)
        x=F.relu(self.up(self.db2(self.d2(x)),t3))+t3; h,w=x.shape[-2:]; x=x.flatten(2).transpose(1,2)
        x=self.fmap(self.dn4(self.dbk2(x,h,w)),b,h,w)
        x=F.relu(self.up(self.db3(self.d3(x)),t2))+t2
        x=F.relu(self.up(self.db4(self.d4(x)),t1))+t1
        x=F.relu(F.interpolate(self.d5(x),size=original,mode='bilinear',align_corners=False))
        return self.final(x)

model=UNext(input_channels=1,num_classes=1).to(DEVICE)
with torch.no_grad(): out=model(images[:2].to(DEVICE))
assert out.shape == masks[:2].shape
print('Forward pass:', out.shape, '| parameters:', sum(p.numel() for p in model.parameters()))


In [ ]:
class BCEDiceLoss(nn.Module):
    def __init__(self,smooth=1e-5): super().__init__(); self.smooth=smooth
    def forward(self,logits,targets):
        bce=F.binary_cross_entropy_with_logits(logits,targets)
        p=torch.sigmoid(logits).flatten(1); t=targets.flatten(1)
        dice=(2*(p*t).sum(1)+self.smooth)/(p.sum(1)+t.sum(1)+self.smooth)
        return 0.5*bce+(1-dice.mean())

@torch.no_grad()
def batch_metrics(logits,targets,threshold=0.5):
    p=(torch.sigmoid(logits)>=threshold).float().flatten(1); t=(targets>=0.5).float().flatten(1)
    tp=(p*t).sum(1); fp=(p*(1-t)).sum(1); fn=((1-p)*t).sum(1); eps=1e-7
    return {'dice':((2*tp+eps)/(2*tp+fp+fn+eps)).mean().item(), 'iou':((tp+eps)/(tp+fp+fn+eps)).mean().item()}

criterion=BCEDiceLoss()
optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
warmup=torch.optim.lr_scheduler.LinearLR(optimizer,start_factor=0.1,total_iters=5)
cosine=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=MAX_EPOCHS-5,eta_min=1e-6)
scheduler=torch.optim.lr_scheduler.SequentialLR(optimizer,[warmup,cosine],milestones=[5])
scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)

def run_epoch(loader,training):
    model.train(training); totals={'loss':0.,'dice':0.,'iou':0.}; seen=0
    for x,y,_ in tqdm(loader,leave=False):
        x,y=x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.amp.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=USE_AMP):
                logits=model(x); loss=criterion(logits,y)
            if training:
                scaler.scale(loss).backward(); scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); scaler.step(optimizer); scaler.update()
        m=batch_metrics(logits.detach(),y); n=x.size(0); seen+=n
        totals['loss']+=loss.item()*n; totals['dice']+=m['dice']*n; totals['iou']+=m['iou']*n
    return {k:v/seen for k,v in totals.items()}

def save_history(history):
    with HISTORY_PATH.open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(history[0])); writer.writeheader(); writer.writerows(history)
print('Training components ready.')


In [ ]:
RESUME_TRAINING = False  # Change to True after a Colab disconnect.
start_epoch,best_dice,stale,history=0,-1.0,0,[]
if RESUME_TRAINING:
    checkpoint=torch.load(RECOVERY_PATH,map_location=DEVICE,weights_only=False)
    model.load_state_dict(checkpoint['model']); optimizer.load_state_dict(checkpoint['optimizer'])
    scheduler.load_state_dict(checkpoint['scheduler']); scaler.load_state_dict(checkpoint['scaler'])
    start_epoch,best_dice,stale,history=checkpoint['epoch'],checkpoint['best_dice'],checkpoint['stale'],checkpoint['history']
    if checkpoint.get('generator_state') is not None: generator.set_state(checkpoint['generator_state'])
    print('Resuming at epoch',start_epoch+1)
else:
    print('Starting fresh training.')


In [ ]:
for epoch in range(start_epoch,MAX_EPOCHS):
    began=time.time(); lr=optimizer.param_groups[0]['lr']
    train=run_epoch(train_loader,True); val=run_epoch(val_loader,False); scheduler.step()
    record={'epoch':epoch+1,'lr':lr,'train_loss':train['loss'],'train_dice':train['dice'],'train_iou':train['iou'],'val_loss':val['loss'],'val_dice':val['dice'],'val_iou':val['iou'],'minutes':(time.time()-began)/60}
    history.append(record); improved=val['dice']>best_dice+1e-4
    if improved: best_dice=val['dice']; stale=0
    else: stale+=1
    state={'epoch':epoch+1,'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'best_dice':best_dice,'stale':stale,'history':history,'generator_state':generator.get_state()}
    torch.save(state,RECOVERY_PATH)
    if improved: torch.save(state,BEST_PATH)
    save_history(history)
    print(f"Epoch {epoch+1:03d} | lr {lr:.2e} | train L {train['loss']:.4f} D {train['dice']:.4f} | val L {val['loss']:.4f} D {val['dice']:.4f} I {val['iou']:.4f} | best {best_dice:.4f} | {record['minutes']:.1f} min")
    if stale>=PATIENCE:
        print('Early stopping.'); break
print('Training complete. Best validation Dice:',best_dice)


In [ ]:
best=torch.load(BEST_PATH,map_location=DEVICE,weights_only=False); model.load_state_dict(best['model']); model.eval()
thresholds=torch.arange(0.20,0.81,0.05,device=DEVICE); tp=torch.zeros_like(thresholds); fp=torch.zeros_like(thresholds); fn=torch.zeros_like(thresholds)
with torch.no_grad():
    for x,y,_ in tqdm(val_loader,desc='Tune threshold'):
        probability=torch.sigmoid(model(x.to(DEVICE))).unsqueeze(0); target=y.to(DEVICE).unsqueeze(0)
        prediction=(probability>=thresholds[:,None,None,None,None]).float()
        tp+=(prediction*target).sum((1,2,3,4)); fp+=(prediction*(1-target)).sum((1,2,3,4)); fn+=((1-prediction)*target).sum((1,2,3,4))
val_dice=(2*tp)/(2*tp+fp+fn+1e-7); best_index=int(val_dice.argmax()); SELECTED_THRESHOLD=float(thresholds[best_index])
print('Selected validation threshold:',SELECTED_THRESHOLD,'| global val Dice:',float(val_dice[best_index]))

@torch.no_grad()
def evaluate_final(loader,threshold):
    losses=[]; per_dice=[]; per_iou=[]; TP=FP=FN=0.0
    for x,y,_ in tqdm(loader,desc='Final test'):
        x,y=x.to(DEVICE),y.to(DEVICE); logits=model(x); losses.append((criterion(logits,y).item(),x.size(0)))
        p=(torch.sigmoid(logits)>=threshold).float(); pf=p.flatten(1); yf=y.flatten(1)
        tpi=(pf*yf).sum(1); fpi=(pf*(1-yf)).sum(1); fni=((1-pf)*yf).sum(1); eps=1e-7
        per_dice.extend(((2*tpi+eps)/(2*tpi+fpi+fni+eps)).cpu().tolist()); per_iou.extend(((tpi+eps)/(tpi+fpi+fni+eps)).cpu().tolist())
        TP+=tpi.sum().item(); FP+=fpi.sum().item(); FN+=fni.sum().item()
    return {'loss':sum(v*n for v,n in losses)/sum(n for _,n in losses),'per_image_dice':float(np.mean(per_dice)),'per_image_iou':float(np.mean(per_iou)),'global_dice':2*TP/(2*TP+FP+FN),'global_iou':TP/(TP+FP+FN),'threshold':threshold,'best_epoch':best['epoch']}

results=evaluate_final(test_loader,SELECTED_THRESHOLD)
with RESULTS_PATH.open('w') as f: json.dump(results,f,indent=2)
print(json.dumps(results,indent=2))


In [ ]:
epochs=[r['epoch'] for r in history]
fig,axes=plt.subplots(1,3,figsize=(17,4))
axes[0].plot(epochs,[r['train_loss'] for r in history],label='train'); axes[0].plot(epochs,[r['val_loss'] for r in history],label='val'); axes[0].set_title('Loss')
axes[1].plot(epochs,[r['train_dice'] for r in history],label='train'); axes[1].plot(epochs,[r['val_dice'] for r in history],label='val'); axes[1].set_title('Dice'); axes[1].set_ylim(0,1)
axes[2].plot(epochs,[r['train_iou'] for r in history],label='train'); axes[2].plot(epochs,[r['val_iou'] for r in history],label='val'); axes[2].set_title('IoU'); axes[2].set_ylim(0,1)
for ax in axes: ax.set_xlabel('Epoch'); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(RUN_DIR/'training_curves.png',dpi=200,bbox_inches='tight'); plt.show()

x,y,meta=next(iter(test_loader)); x=x.to(DEVICE)
with torch.no_grad(): probability=torch.sigmoid(model(x)).cpu(); prediction=(probability>=SELECTED_THRESHOLD).float()
fig,axes=plt.subplots(4,4,figsize=(14,14))
for i in range(4):
    image=x[i,0].cpu(); truth=y[i,0]; prob=probability[i,0]; pred=prediction[i,0]
    axes[i,0].imshow(image,cmap='gray'); axes[i,0].set_title(meta['img_id'][i],fontsize=7)
    axes[i,1].imshow(truth,cmap='gray'); axes[i,1].set_title('Ground truth')
    axes[i,2].imshow(prob,cmap='viridis',vmin=0,vmax=1); axes[i,2].set_title('Probability')
    axes[i,3].imshow(image,cmap='gray'); axes[i,3].imshow(pred,cmap='Reds',alpha=.45); axes[i,3].set_title('Prediction')
    for ax in axes[i]: ax.axis('off')
plt.tight_layout(); plt.savefig(RUN_DIR/'test_predictions.png',dpi=200,bbox_inches='tight'); plt.show()


## Interpretation

Use `per_image_dice` as the closest comparison with the first experiment, and also report `global_dice` so aggregation is explicit. The probability threshold is selected using validation data only, then frozen before the test set is evaluated. This notebook is an improvement experiment, not an exact replication of the paper.